This notebook performs alpha motor neuron subtype label transfer from the multiome RNA data to the RNA-only data and also performs pseudobulk differential expression analyses using the multiome RNA data.

## Setup

In [1]:
# Set paths
SOD1_RENV_RNA="/oak/stanford/groups/agitler/Shared/Shared_Jupyter_Notebook_Analysis/4.1.1-OG-RNA/"
renv::load(SOD1_RENV_RNA)

In [2]:
# Load packages
suppressPackageStartupMessages({
  library(Seurat)
  library(SoupX)
  library(ggplot2)
  library(dplyr)
  library(BiocParallel)
  library(scDblFinder)
  library(svglite)
  library(pheatmap)
  library(matrixStats)
  library(tibble)
  library(DESeq2)
  library(VennDiagram)
  library(grid)
  library(stringr)
  library(ggrepel)
  library(ggrastr)
  library(ggpubr)
  library(forcats)
  library(ggbreak)
  library(cowplot)
  library(colorspace)
  library(broom)
  library(colorspace)
  library(ArchR)
})


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .______      
          /   \     |   _ 

## Directory to save output

In [3]:
save_dir <- '/oak/stanford/groups/agitler/Shared/SOD1_Paper/RNA/'

## Make Seurat object from multiome RNA data

In [31]:
#Load ArchR Project for alpha MNs
proj_all_dr <- loadArchRProject('/oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_all_dr')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [32]:
# Extract metadata
metadata <- getCellColData(proj_all_dr) %>% as.data.frame()

In [33]:
#Get Input RNA Files
sample1_rna <- '/oak/stanford/groups/agitler/Olivia/single_cell_data/10X_analyzed/SOD1_multiome/5_25_sample1/outs/filtered_feature_bc_matrix.h5'
sample2_rna <- '/oak/stanford/groups/agitler/Olivia/single_cell_data/10X_analyzed/SOD1_multiome/6_3_sample1/outs/filtered_feature_bc_matrix.h5'
sample3_rna <- '/oak/stanford/groups/agitler/Olivia/single_cell_data/10X_analyzed/SOD1_multiome/6_3_sample2/outs/filtered_feature_bc_matrix.h5'
sample4_rna <- '/oak/stanford/groups/agitler/Olivia/single_cell_data/10X_analyzed/SOD1_multiome/12_22_sample1/outs/filtered_feature_bc_matrix.h5'
sample5_rna <- '/oak/stanford/groups/agitler/Olivia/single_cell_data/10X_analyzed/SOD1_multiome/12_22_sample2/outs/filtered_feature_bc_matrix.h5'

inputFiles_rna <- c(sample1_5_25=sample1_rna, sample1_6_3=sample2_rna, sample2_6_3=sample3_rna, sample1_12_22=sample4_rna, sample2_12_22=sample5_rna)

In [34]:
# Import scRNA
seRNA <- import10xFeatureMatrix(
    input = inputFiles_rna,
    names = names(inputFiles_rna),
    strictMatch = FALSE
)

Importing Feature Matrix 1 of 5

Importing Feature Matrix 2 of 5

Importing Feature Matrix 3 of 5

Importing Feature Matrix 4 of 5

Importing Feature Matrix 5 of 5

Re-ordering RNA matricies for consistency.

Merging individual RNA objects...


Merging sample1_6_3

Warning! Some values within column "id" of the rowData (gene metadata) of your objects do not precisely match!

This is often caused by slight variations in Ensembl IDs and gene locations used by cellranger across different samples. ArchR will ignore these mismatches and allow merging to proceed but you should check to make sure that these are ok for your data.


Mismatch in column "id" row 25927 for sample1_6_3: ENSMUSG00000116160 does not exactly match ENSMUSG00000116253!

strictMatch = FALSE so mismatching information will be coerced to match the first sample provided.


Merging sample2_6_3

Warning! Some values within column "id" of the rowData (gene metadata) of your objects do not precisely match!

This is often caused

In [35]:
# Get the counts data as a sparse matrix
counts_matrix <- assay(seRNA, "counts")

# Extract the column names of the sparse matrix
nuclei_names <- colnames(counts_matrix)

# Filter the column names based on whether they are in the row names of metadata
filtered_nuclei <- nuclei_names[nuclei_names %in% rownames(metadata)]

# Subset the sparse matrix to retain only the filtered columns
filtered_counts_matrix <- counts_matrix[, filtered_nuclei]

In [36]:
# Create the Seurat object
proj_all_dr_multiome_rna_sobj <- CreateSeuratObject(counts = filtered_counts_matrix)

# Add metadata to the Seurat object
proj_all_dr_multiome_rna_sobj <- AddMetaData(object = proj_all_dr_multiome_rna_sobj, metadata = metadata)

# Verify the Seurat object
proj_all_dr_multiome_rna_sobj

An object of class Seurat 
32233 features across 26965 samples within 1 assay 
Active assay: RNA (32233 features, 0 variable features)

In [37]:
proj_all_dr_multiome_rna_sobj@meta.data$orig.ident <- proj_all_dr_multiome_rna_sobj@meta.data$Sample

In [38]:
saveRDS(proj_all_dr_multiome_rna_sobj, file = paste0(save_dir, 'rds_files/proj_all_dr_multiome_rna_sobj.rds'))

## Alpha MN subtype label transfer from multiome to RNA-only alpha MNs

In [5]:
#Load ArchR Project for alpha MNs
proj_alpha <- loadArchRProject('/oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_alpha')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [6]:
# Extract metadata
metadata <- getCellColData(proj_alpha) %>% as.data.frame()

In [7]:
#Get Input RNA Files
sample1_rna <- '/oak/stanford/groups/agitler/Olivia/single_cell_data/10X_analyzed/SOD1_multiome/5_25_sample1/outs/filtered_feature_bc_matrix.h5'
sample2_rna <- '/oak/stanford/groups/agitler/Olivia/single_cell_data/10X_analyzed/SOD1_multiome/6_3_sample1/outs/filtered_feature_bc_matrix.h5'
sample3_rna <- '/oak/stanford/groups/agitler/Olivia/single_cell_data/10X_analyzed/SOD1_multiome/6_3_sample2/outs/filtered_feature_bc_matrix.h5'
sample4_rna <- '/oak/stanford/groups/agitler/Olivia/single_cell_data/10X_analyzed/SOD1_multiome/12_22_sample1/outs/filtered_feature_bc_matrix.h5'
sample5_rna <- '/oak/stanford/groups/agitler/Olivia/single_cell_data/10X_analyzed/SOD1_multiome/12_22_sample2/outs/filtered_feature_bc_matrix.h5'

inputFiles_rna <- c(sample1_5_25=sample1_rna, sample1_6_3=sample2_rna, sample2_6_3=sample3_rna, sample1_12_22=sample4_rna, sample2_12_22=sample5_rna)

In [8]:
# Import scRNA
seRNA <- import10xFeatureMatrix(
    input = inputFiles_rna,
    names = names(inputFiles_rna),
    strictMatch = FALSE
)

Importing Feature Matrix 1 of 5

Importing Feature Matrix 2 of 5

Importing Feature Matrix 3 of 5

Importing Feature Matrix 4 of 5

Importing Feature Matrix 5 of 5

Re-ordering RNA matricies for consistency.

Merging individual RNA objects...


Merging sample1_6_3

Warning! Some values within column "id" of the rowData (gene metadata) of your objects do not precisely match!

This is often caused by slight variations in Ensembl IDs and gene locations used by cellranger across different samples. ArchR will ignore these mismatches and allow merging to proceed but you should check to make sure that these are ok for your data.


Mismatch in column "id" row 25927 for sample1_6_3: ENSMUSG00000116160 does not exactly match ENSMUSG00000116253!

strictMatch = FALSE so mismatching information will be coerced to match the first sample provided.


Merging sample2_6_3

Warning! Some values within column "id" of the rowData (gene metadata) of your objects do not precisely match!

This is often caused

In [9]:
# Get the counts data as a sparse matrix
counts_matrix <- assay(seRNA, "counts")

# Extract the column names of the sparse matrix
nuclei_names <- colnames(counts_matrix)

# Filter the column names based on whether they are in the row names of metadata
filtered_nuclei <- nuclei_names[nuclei_names %in% rownames(metadata)]

# Subset the sparse matrix to retain only the filtered columns
filtered_counts_matrix <- counts_matrix[, filtered_nuclei]

In [10]:
# Create the Seurat object
alpha_multiome_rna_sobj <- CreateSeuratObject(counts = filtered_counts_matrix)

# Add metadata to the Seurat object
alpha_multiome_rna_sobj <- AddMetaData(object = alpha_multiome_rna_sobj, metadata = metadata)

# Verify the Seurat object
alpha_multiome_rna_sobj

An object of class Seurat 
32233 features across 3433 samples within 1 assay 
Active assay: RNA (32233 features, 0 variable features)

In [11]:
alpha_multiome_rna_sobj@meta.data$orig.ident <- alpha_multiome_rna_sobj@meta.data$Sample
alpha_multiome_rna_sobj <- NormalizeData(alpha_multiome_rna_sobj)
alpha_multiome_rna_sobj <- FindVariableFeatures(alpha_multiome_rna_sobj, selection.method = "vst", nfeatures = 2000)

In [12]:
saveRDS(alpha_multiome_rna_sobj, file = paste0(save_dir, 'rds_files/alpha_multiome_rna_sobj.rds'))

In [13]:
# Load alpha MN RNA Seurat object
alpha_subset <- readRDS(file = paste0(save_dir, 'rds_files/alpha_subset.rds'))

In [14]:
alpha.anchors <- FindTransferAnchors(reference = alpha_multiome_rna_sobj, query = alpha_subset, 
    reference.assay = "RNA", query.assay = "RNA", dims = 1:50)
predictions <- TransferData(anchorset = alpha.anchors, refdata = alpha_multiome_rna_sobj$alpha_subtype, 
    dims = 1:50)
alpha_label_transfer_50 <- AddMetaData(alpha_subset, metadata = predictions)

Warning message:
“npcs is smaller than the largest value requested by the dims parameter.
Setting npcs to 50 and continuing.”
Performing PCA on the provided reference using 1898 features as input.

Projecting cell embeddings

Finding neighborhoods

Finding anchors

	Found 7146 anchors

Filtering anchors

	Retained 5268 anchors

Finding integration vectors

Finding integration vector weights

Predicting cell labels



In [15]:
saveRDS(alpha_label_transfer_50, file = paste0(save_dir, 'rds_files/alpha_label_transfer_50.rds'))

## DESeq2 differential expression analysis helper function

In [4]:
prepare_counts <- function(seurat_object, test.category, by.category, min.nuclei) {
    seurat_object$samples <- paste0(seurat_object@meta.data[, test.category], "_", seurat_object$orig.ident)
    
    # Aggregate counts for each sample
    counts <- AggregateExpression(seurat_object, 
                    group.by = c(by.category, "samples"),
                    assays = "RNA",
                    slot = "counts",
                    return.seurat = FALSE)
    
    counts <- counts$RNA
    
    # Transpose counts and convert to data frame
    counts.t.df <- as.data.frame(t(counts))
    
    # Get samples to delete (number of nuclei < min.nuclei)
    # 1. Get table containing nuclei counts for each sample/by.category pair
    nuclei_count_table <- table(seurat_object@meta.data[, by.category], seurat_object$samples)
    
    # 2. Get row names (by.category) and column names (samples)
    by_category <- rownames(nuclei_count_table)
    samples <- colnames(nuclei_count_table)

    # 3. Iterate over all combinations, check the condition, and store samples to delete in the vector
    to_delete <- c()

    for (i in by_category) {
      for (j in samples) {
        value <- nuclei_count_table[i, j]
        if (value < min.nuclei) {
          to_delete <- c(to_delete, paste(i, j, sep = "_"))
        }
      }
    }
    
    # 4. Filter out sample/by.category pairs with fewer than min.nuclei 
    counts.t.df <- counts.t.df %>% filter(!row.names(.) %in% to_delete)
    
    # Split data frame by by.category
    splitRows <- sub("_.*", "", rownames(counts.t.df))
    counts.split <- split.data.frame(counts.t.df, f = factor(splitRows)) 
    
    # For every element in counts.split, remove by.category from the row names and then transpose
    counts.split <- lapply(counts.split, function(x){
    rownames(x) <- sub("^[^_]*_", "", rownames(x))
    t(x)
    })
    
}

## Multiome RNA DESeq2 analyses

### DM/DAMN vs. nonDM/DAMN from the multiome RNA data
alpha_multiome_RNA_nonDAMN_vs_DAMN.csv

In [5]:
alpha_multiome_rna_sobj <- readRDS(file = paste0(save_dir, 'rds_files/alpha_multiome_rna_sobj.rds'))

In [6]:
alpha_multiome_rna_sobj@meta.data

,orig.ident,nCount_RNA,nFeature_RNA,Sample,TSSEnrichment,ReadsInTSS,ReadsInPromoter,ReadsInBlacklist,PromoterRatio,PassQC,⋯,Gex_RiboRatio,Stage,Clusters,cell_class,cell_class_UMAP_labels,cholinergic_type,cholinergic_type_UMAP_labels,alpha_subtype,ReadsInPeaks,FRIP
,<chr>,<dbl>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
sample1_5_25#AAACAGCCAATAATGG-1,sample1_5_25,19067,5034,sample1_5_25,9.778,1195,4790,570,0.12498043,1,⋯,0,control,C5,Cholinergic Neurons,Cholinergic Neurons,Alpha MNs,Alpha MNs,Slow-Firing,14318,0.3736040
sample1_5_25#AAACAGCCAGTTGCGT-1,sample1_5_25,29913,5928,sample1_5_25,9.022,811,3357,376,0.12552348,1,⋯,0,control,C5,Cholinergic Neurons,Cholinergic Neurons,Alpha MNs,Alpha MNs,Slow-Firing,9676,0.3618008
sample1_5_25#AAACAGCCATGTGGGA-1,sample1_5_25,35529,6593,sample1_5_25,6.371,1332,6123,1049,0.08965123,1,⋯,0,mid-late,C3,Cholinergic Neurons,Cholinergic Neurons,Alpha MNs,Alpha MNs,Fast-Firing,21909,0.3208512
sample1_5_25#AAACCGCGTTTAACGG-1,sample1_5_25,40108,6863,sample1_5_25,8.537,1733,7200,1083,0.10369858,1,⋯,0,mid-late,C1,Cholinergic Neurons,Cholinergic Neurons,Alpha MNs,Alpha MNs,Late DAMN,24815,0.3574824
sample1_5_25#AAACCGGCAGCTTACA-1,sample1_5_25,39561,6650,sample1_5_25,7.882,1405,5910,742,0.10378618,1,⋯,0,mid-late,C2,Cholinergic Neurons,Cholinergic Neurons,Alpha MNs,Alpha MNs,Intermediate,20797,0.3653468
sample1_5_25#AAACGTACAAGTGAAC-1,sample1_5_25,19601,4959,sample1_5_25,6.978,74,262,29,0.11302847,1,⋯,0,control,C4,Cholinergic Neurons,Cholinergic Neurons,Alpha MNs,Alpha MNs,Early DAMN,808,0.3485764
sample1_5_25#AAAGCAAGTAAGCTCA-1,sample1_5_25,14612,4430,sample1_5_25,8.113,1057,4764,575,0.10769996,1,⋯,0,control,C5,Cholinergic Neurons,Cholinergic Neurons,Alpha MNs,Alpha MNs,Slow-Firing,13603,0.3075793
sample1_5_25#AAAGCAAGTTGTGATG-1,sample1_5_25,8117,3276,sample1_5_25,8.614,87,322,38,0.13597973,1,⋯,0,mid-late,C5,Cholinergic Neurons,Cholinergic Neurons,Alpha MNs,Alpha MNs,Slow-Firing,1057,0.4463682
sample1_5_25#AAAGCACCACAACCTA-1,sample1_5_25,23990,5518,sample1_5_25,8.289,1352,5975,968,0.10995988,1,⋯,0,mid-late,C2,Cholinergic Neurons,Cholinergic Neurons,Alpha MNs,Alpha MNs,Intermediate,19327,0.3557859


In [8]:
alpha_multiome_rna_sobj$damn_status <- ifelse(alpha_multiome_rna_sobj$alpha_subtype %in% c("Early DAMN", "Late DAMN"), "DAMN", "nonDAMN")

In [9]:
# Prepare counts for the Seurat object
alpha.counts.split <- prepare_counts(alpha_multiome_rna_sobj, test.category='damn_status', 
                                           by.category='damn_status', min.nuclei = 50)

In [10]:
counts <- cbind(alpha.counts.split[[1]], alpha.counts.split[[2]])

In [12]:
head(counts)

,DAMN_sample1_12_22,DAMN_sample1_5_25,DAMN_sample1_6_3,DAMN_sample2_12_22,DAMN_sample2_6_3,nonDAMN_sample1_12_22,nonDAMN_sample1_5_25,nonDAMN_sample1_6_3,nonDAMN_sample2_12_22,nonDAMN_sample2_6_3
Gm1992,24,82,127,39,54,207,304,129,625,145
Gm19938,51,189,176,38,162,539,736,184,444,343
Xkr4,1809,6426,5030,1327,3842,19982,27932,5308,15123,8409
Gm37381,0,0,1,1,0,0,1,0,0,1
Rp1,1,2,2,3,3,9,16,3,10,11
Sox17,0,0,0,0,0,0,0,0,0,0


In [13]:
# Create a data frame of sample information
colData <- data.frame(samples = colnames(counts))
colData

samples
<chr>
DAMN_sample1_12_22
DAMN_sample1_5_25
DAMN_sample1_6_3
DAMN_sample2_12_22
DAMN_sample2_6_3
nonDAMN_sample1_12_22
nonDAMN_sample1_5_25
nonDAMN_sample1_6_3
nonDAMN_sample2_12_22


In [14]:
# Add a condition column containing the test.category information (ex: cell class) for each sample
colData$condition <- sub("_.*", "", colData$samples) 
colData

samples,condition
<chr>,<chr>
DAMN_sample1_12_22,DAMN
DAMN_sample1_5_25,DAMN
DAMN_sample1_6_3,DAMN
DAMN_sample2_12_22,DAMN
DAMN_sample2_6_3,DAMN
nonDAMN_sample1_12_22,nonDAMN
nonDAMN_sample1_5_25,nonDAMN
nonDAMN_sample1_6_3,nonDAMN
nonDAMN_sample2_12_22,nonDAMN


In [15]:
# Convert the sample names to row names
colData <- colData %>% column_to_rownames(var = "samples")
colData

,condition
,<chr>
DAMN_sample1_12_22,DAMN
DAMN_sample1_5_25,DAMN
DAMN_sample1_6_3,DAMN
DAMN_sample2_12_22,DAMN
DAMN_sample2_6_3,DAMN
nonDAMN_sample1_12_22,nonDAMN
nonDAMN_sample1_5_25,nonDAMN
nonDAMN_sample1_6_3,nonDAMN
nonDAMN_sample2_12_22,nonDAMN


In [16]:
# Create DESEq object
dds <- DESeqDataSetFromMatrix(countData = counts,
                      colData = colData,
                      design = ~ condition)

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”


In [17]:
# Run DESeq
dds <- DESeq(dds, minReplicatesForReplace = Inf)

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



In [18]:
results_name <- resultsNames(dds)
results_name <- results_name[grep("^condition", results_name)]
results_name

[1] "condition_nonDAMN_vs_DAMN"

In [19]:
savedir.path <- paste0(save_dir, 'DESeq2/')

res <- results(dds, name = results_name) %>% data.frame(.) 
res$gene <- rownames(res)
res <- res %>% arrange(padj) %>% filter(!is.na(padj))
rownames(res) <- res$gene
res <- dplyr::select(res, -gene)
write.csv(res, file=paste0(savedir.path, "alpha_multiome_RNA_", sub("^[^_]*_", "", results_name), '.csv'))

### DAI vs. Non-cholinergic interneurons from multiome RNA data
multiome_RNA_Non.Cholinergic.Interneurons_vs_Disease.Associated.Interneurons.csv

In [39]:
proj_all_dr_multiome_rna_sobj <- readRDS(file = paste0(save_dir, 'rds_files/proj_all_dr_multiome_rna_sobj.rds'))

In [40]:
dai_cholInt_multiome_rna_sobj <- subset(proj_all_dr_multiome_rna_sobj, subset = cell_class %in% c('Disease-Associated Interneurons','Non-Cholinergic Interneurons'))

In [53]:
# Prepare counts for the Seurat object
counts.split <- prepare_counts(dai_cholInt_multiome_rna_sobj, test.category='cell_class', 
                                           by.category='cell_class', min.nuclei = 15)

In [54]:
counts <- cbind(counts.split[[1]], counts.split[[2]])

In [55]:
head(counts)

,Disease-Associated Interneurons_sample1_5_25,Disease-Associated Interneurons_sample1_6_3,Disease-Associated Interneurons_sample2_6_3,Non-Cholinergic Interneurons_sample1_12_22,Non-Cholinergic Interneurons_sample1_5_25,Non-Cholinergic Interneurons_sample1_6_3,Non-Cholinergic Interneurons_sample2_12_22,Non-Cholinergic Interneurons_sample2_6_3
Gm1992,3,7,5,100,131,300,239,322
Gm19938,3,3,29,260,456,496,250,908
Xkr4,241,167,680,9031,17308,12208,7908,20546
Gm37381,0,0,0,0,1,0,0,0
Rp1,0,0,0,5,3,7,3,8
Sox17,0,0,0,0,0,0,0,0


In [56]:
# Create a data frame of sample information
colData <- data.frame(samples = colnames(counts))
colData

samples
<chr>
Disease-Associated Interneurons_sample1_5_25
Disease-Associated Interneurons_sample1_6_3
Disease-Associated Interneurons_sample2_6_3
Non-Cholinergic Interneurons_sample1_12_22
Non-Cholinergic Interneurons_sample1_5_25
Non-Cholinergic Interneurons_sample1_6_3
Non-Cholinergic Interneurons_sample2_12_22
Non-Cholinergic Interneurons_sample2_6_3


In [57]:
# Add a condition column containing the test.category information (ex: cell class) for each sample
colData$condition <- sub("_.*", "", colData$samples) 
colData

samples,condition
<chr>,<chr>
Disease-Associated Interneurons_sample1_5_25,Disease-Associated Interneurons
Disease-Associated Interneurons_sample1_6_3,Disease-Associated Interneurons
Disease-Associated Interneurons_sample2_6_3,Disease-Associated Interneurons
Non-Cholinergic Interneurons_sample1_12_22,Non-Cholinergic Interneurons
Non-Cholinergic Interneurons_sample1_5_25,Non-Cholinergic Interneurons
Non-Cholinergic Interneurons_sample1_6_3,Non-Cholinergic Interneurons
Non-Cholinergic Interneurons_sample2_12_22,Non-Cholinergic Interneurons
Non-Cholinergic Interneurons_sample2_6_3,Non-Cholinergic Interneurons


In [58]:
# Convert the sample names to row names
colData <- colData %>% column_to_rownames(var = "samples")
colData

,condition
,<chr>
Disease-Associated Interneurons_sample1_5_25,Disease-Associated Interneurons
Disease-Associated Interneurons_sample1_6_3,Disease-Associated Interneurons
Disease-Associated Interneurons_sample2_6_3,Disease-Associated Interneurons
Non-Cholinergic Interneurons_sample1_12_22,Non-Cholinergic Interneurons
Non-Cholinergic Interneurons_sample1_5_25,Non-Cholinergic Interneurons
Non-Cholinergic Interneurons_sample1_6_3,Non-Cholinergic Interneurons
Non-Cholinergic Interneurons_sample2_12_22,Non-Cholinergic Interneurons
Non-Cholinergic Interneurons_sample2_6_3,Non-Cholinergic Interneurons


In [59]:
# Create DESEq object
dds <- DESeqDataSetFromMatrix(countData = counts,
                      colData = colData,
                      design = ~ condition)

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]



In [60]:
# Run DESeq
dds <- DESeq(dds, minReplicatesForReplace = Inf)

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

final dispersion estimates

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This 

In [61]:
results_name <- resultsNames(dds)
results_name <- results_name[grep("^condition", results_name)]
results_name

[1] "condition_Non.Cholinergic.Interneurons_vs_Disease.Associated.Interneurons"

In [62]:
savedir.path <- paste0(save_dir, 'DESeq2/')

res <- results(dds, name = results_name) %>% data.frame(.) 
res$gene <- rownames(res)
res <- res %>% arrange(padj) %>% filter(!is.na(padj))
rownames(res) <- res$gene
res <- dplyr::select(res, -gene)
write.csv(res, file=paste0(savedir.path, "multiome_RNA_", sub("^[^_]*_", "", results_name), '.csv'))